# DR9 local overdensity: sweep-file accumulation

This notebook is the RAM-safe sweep version of the DR9 local-overdensity calculation. It can still run one-file diagnostics, but its main production section iterates through DR9 sweep files one at a time, accumulates galaxy counts and sweep-footprint coverage, and computes a local-background-subtracted overdensity around redMaPPer clusters.

In [ ]:
# Configuration: edit these values if needed, then run the notebook top to bottom.

from pathlib import Path
import sys

# On NERSC this should make your repo-level setup.py importable.
REPO_DIR = Path('/global/homes/z/zzhang13/DESI')
if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    from setup import data_dir
    CATALOG_PATH = Path(data_dir()) / 'bgs_clus_RM_gal_matched.pickle'
except Exception:
    CATALOG_PATH = Path('/global/homes/z/zzhang13/DESI/data/bgs_clus_RM_gal_matched.pickle')

SWEEP_DIRS = [
    Path('/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/north/sweep/9.0'),
    Path('/global/cfs/cdirs/cosmo/data/legacysurvey/dr9/south/sweep/9.0'),
]
SWEEP_PATTERN = 'sweep-*.fits'
MAX_SWEEP_FILES = None   # set to e.g. 5 for a quick test
SAVE_EVERY_N_FILES = 10
RANDOM_FILE = Path('/global/cfs/cdirs/desi/target/catalogs/dr9/0.49.0/randoms/resolve/randoms-10-0.fits')

OUTPUT_DIR = REPO_DIR / 'local_overdensity' / 'dr9_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RMIN_HMPC = 1.5
RMAX_HMPC = 8.0        # signal annulus: cluster environment
BG_RMIN_HMPC = 12.0    # local background annulus
BG_RMAX_HMPC = 20.0
COVERAGE_MIN = 0.8     # require this coverage in both signal and background annuli
R_MAG_LIMIT = 23.5

NSIDE_GAL = 4096
NSIDE_RAND = 1024
NSIDE_RAND_AREA = 128
NSIDE_SWEEP_COVERAGE = 1024

print('Catalog:', CATALOG_PATH)
print('Sweep dirs:')
for sweep_dir in SWEEP_DIRS:
    print('  ', sweep_dir)
print('Random file:', RANDOM_FILE)
print('Output dir:', OUTPUT_DIR.resolve())


In [ ]:
import gc
import pickle
from glob import glob

import astropy.units as u
from astropy.cosmology import Planck18
from astropy.table import Table, unique
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binned_statistic, pearsonr, spearmanr

plt.rcParams.update({
    'figure.constrained_layout.use': True,
    'font.size': 13,
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

## Helper functions

In [ ]:
def nanomaggies_to_mag(flux):
    flux = np.asarray(flux, dtype=float)
    mag = np.full(flux.shape, np.nan, dtype=float)
    good = np.isfinite(flux) & (flux > 0)
    mag[good] = 22.5 - 2.5 * np.log10(flux[good])
    return mag


def dereddened_mag(table, band):
    band = band.upper()
    flux = np.asarray(table[f'FLUX_{band}'], dtype=float)
    transmission = np.asarray(table[f'MW_TRANSMISSION_{band}'], dtype=float)
    return nanomaggies_to_mag(flux / transmission)


def dr9_galaxy_mask(table, r_mag_limit=23.5):
    mask = np.isfinite(table['RA']) & np.isfinite(table['DEC'])
    if 'TYPE' in table.colnames:
        mask &= np.asarray(table['TYPE']) != 'PSF'
    for band in ('G', 'R', 'Z'):
        col = f'FLUX_IVAR_{band}'
        if col in table.colnames:
            mask &= np.asarray(table[col], dtype=float) > 0
    if r_mag_limit is not None:
        r_dered = dereddened_mag(table, 'R')
        mask &= np.isfinite(r_dered) & (r_dered < r_mag_limit)
    return np.asarray(mask, dtype=bool)


def read_one_dr9_sweep(filename, r_mag_limit=23.5):
    table = Table.read(filename, hdu=1, memmap=True)
    mask = dr9_galaxy_mask(table, r_mag_limit=r_mag_limit)
    out = table[mask][['RA', 'DEC', 'TYPE']].copy()
    out['r_dered'] = dereddened_mag(table[mask], 'R')
    return out


def read_random_positions(filename):
    randoms = Table.read(filename, hdu=1, memmap=True)
    ra = np.asarray(randoms['RA'], dtype=float)
    dec = np.asarray(randoms['DEC'], dtype=float)
    good = np.isfinite(ra) & np.isfinite(dec)
    return ra[good], dec[good]

def read_sweep_bounds(filename):
    table = Table.read(filename, hdu=1, memmap=True)
    ra = np.asarray(table['RA'], dtype=float)
    dec = np.asarray(table['DEC'], dtype=float)
    good = np.isfinite(ra) & np.isfinite(dec)
    return {
        'ra_min': np.nanmin(ra[good]),
        'ra_max': np.nanmax(ra[good]),
        'dec_min': np.nanmin(dec[good]),
        'dec_max': np.nanmax(dec[good]),
    }


def inside_ra_dec_box(ra, dec, bounds):
    ra = np.asarray(ra, dtype=float)
    dec = np.asarray(dec, dtype=float)
    return (
        (ra >= bounds['ra_min'])
        & (ra <= bounds['ra_max'])
        & (dec >= bounds['dec_min'])
        & (dec <= bounds['dec_max'])
    )


In [ ]:
def radec_to_unitvec(ra_deg, dec_deg):
    ra = np.deg2rad(np.asarray(ra_deg, dtype=float))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=float))
    cosd = np.cos(dec)
    return np.column_stack((cosd * np.cos(ra), cosd * np.sin(ra), np.sin(dec)))


def annulus_theta_radians(z, Rmin_hMpc=1.5, Rmax_hMpc=10.0, cosmo=Planck18):
    h = cosmo.h
    Rmin_Mpc = (Rmin_hMpc / h) * u.Mpc
    Rmax_Mpc = (Rmax_hMpc / h) * u.Mpc
    Dm = cosmo.comoving_transverse_distance(z)
    th_min = (Rmin_Mpc / Dm).decompose().value
    th_max = (Rmax_Mpc / Dm).decompose().value
    return th_min, th_max


def spherical_annulus_area_sr(theta_min, theta_max):
    return 2.0 * np.pi * (np.cos(theta_min) - np.cos(theta_max))


def build_galaxy_healpix_index(ra_gal, dec_gal, nside=4096, nest=False):
    theta = np.deg2rad(90.0 - np.asarray(dec_gal, dtype=float))
    phi = np.deg2rad(np.asarray(ra_gal, dtype=float))
    pix = hp.ang2pix(nside, theta, phi, nest=nest)
    order = np.argsort(pix)
    pix_sorted = pix[order]
    uniq_pix, start = np.unique(pix_sorted, return_index=True)
    end = np.r_[start[1:], len(pix_sorted)]
    return {'nside': nside, 'nest': nest, 'order': order, 'uniq_pix': uniq_pix, 'start': start, 'end': end}


def gather_candidates(index, pix_list):
    uniq_pix = index['uniq_pix']
    pix_list = np.asarray(pix_list, dtype=uniq_pix.dtype)
    pos = np.searchsorted(uniq_pix, pix_list)
    inside = pos < len(uniq_pix)
    pos_valid = pos[inside]
    pix_valid = pix_list[inside]
    good = uniq_pix[pos_valid] == pix_valid
    pos = pos_valid[good]
    if len(pos) == 0:
        return np.empty(0, dtype=np.int64)
    order = index['order']
    start = index['start']
    end = index['end']
    return np.concatenate([order[start[p]:end[p]] for p in pos])

In [ ]:
def count_galaxies_in_annuli(
    ra_cl,
    dec_cl,
    z_cl,
    ra_gal,
    dec_gal,
    Rmin_hMpc=1.5,
    Rmax_hMpc=10.0,
    nside=4096,
    nest=False,
    verbose_every=500,
):
    ra_cl = np.asarray(ra_cl, dtype=float)
    dec_cl = np.asarray(dec_cl, dtype=float)
    z_cl = np.asarray(z_cl, dtype=float)
    gal_vecs = radec_to_unitvec(ra_gal, dec_gal)
    cl_vecs = radec_to_unitvec(ra_cl, dec_cl)
    index = build_galaxy_healpix_index(ra_gal, dec_gal, nside=nside, nest=nest)

    counts = np.zeros(len(ra_cl), dtype=np.int64)
    area_sr = np.full(len(ra_cl), np.nan, dtype=float)

    for i, (vec, z) in enumerate(zip(cl_vecs, z_cl)):
        if verbose_every and (i + 1) % verbose_every == 0:
            print(f'  counted {i + 1:,}/{len(ra_cl):,} clusters')

        if not np.isfinite(z) or z <= 0:
            continue

        th_min, th_max = annulus_theta_radians(z, Rmin_hMpc=Rmin_hMpc, Rmax_hMpc=Rmax_hMpc)
        area_sr[i] = spherical_annulus_area_sr(th_min, th_max)

        pix_outer = hp.query_disc(nside, vec, th_max, inclusive=True, nest=nest)
        cand = gather_candidates(index, pix_outer)
        if len(cand) == 0:
            continue

        dot = np.clip(gal_vecs[cand] @ vec, -1.0, 1.0)
        sep = np.arccos(dot)
        counts[i] = np.count_nonzero((sep >= th_min) & (sep < th_max))

    return counts, area_sr


def estimate_random_surface_density(ra_rand, dec_rand, nside_area=128):
    theta = np.deg2rad(90.0 - dec_rand)
    phi = np.deg2rad(ra_rand)
    pix = hp.ang2pix(nside_area, theta, phi)
    n_occ = len(np.unique(pix))
    area_sr = n_occ * hp.nside2pixarea(nside_area)
    area_deg2 = area_sr * (180.0 / np.pi) ** 2
    nbar_sr = len(ra_rand) / area_sr
    nbar_deg2 = len(ra_rand) / area_deg2
    return nbar_sr, nbar_deg2, area_sr, area_deg2

def sweep_annulus_coverage_for_clusters(
    ra_cl,
    dec_cl,
    z_cl,
    sweep_bounds,
    Rmin_hMpc=1.5,
    Rmax_hMpc=8.0,
    nside=1024,
    nest=False,
    verbose_every=500,
):
    """Estimate the fraction of each cluster annulus inside the one sweep file.

    The annulus is represented by equal-area HEALPix pixels. Coverage is the
    fraction of annulus-pixel centers whose RA/Dec lies inside the sweep file
    RA/Dec bounds. This is the relevant one-file footprint cut; it is not a
    DESI random-catalog footprint cut.
    """

    ra_cl = np.asarray(ra_cl, dtype=float)
    dec_cl = np.asarray(dec_cl, dtype=float)
    z_cl = np.asarray(z_cl, dtype=float)
    cl_vecs = radec_to_unitvec(ra_cl, dec_cl)
    coverage = np.full(len(ra_cl), np.nan, dtype=float)
    n_pix_annulus = np.zeros(len(ra_cl), dtype=int)

    for i, (vec, z) in enumerate(zip(cl_vecs, z_cl)):
        if verbose_every and (i + 1) % verbose_every == 0:
            print(f'  coverage {i + 1:,}/{len(ra_cl):,} clusters')

        if not np.isfinite(z) or z <= 0:
            continue

        th_min, th_max = annulus_theta_radians(z, Rmin_hMpc=Rmin_hMpc, Rmax_hMpc=Rmax_hMpc)
        pix_out = hp.query_disc(nside, vec, th_max, inclusive=True, nest=nest)
        pix_in = hp.query_disc(nside, vec, th_min, inclusive=True, nest=nest)
        pix_ann = np.setdiff1d(pix_out, pix_in, assume_unique=False)
        n_pix_annulus[i] = len(pix_ann)

        if len(pix_ann) == 0:
            continue

        theta_pix, phi_pix = hp.pix2ang(nside, pix_ann, nest=nest)
        ra_pix = np.rad2deg(phi_pix)
        dec_pix = 90.0 - np.rad2deg(theta_pix)
        inside = inside_ra_dec_box(ra_pix, dec_pix, sweep_bounds)
        coverage[i] = np.count_nonzero(inside) / len(pix_ann)

    return coverage, n_pix_annulus


## Load Cluster Catalog

Load the redMaPPer/BGS-matched cluster catalog once. The all-sweep sections below stream DR9 sweep files one at a time, so no full DR9 photometric catalog is kept in memory.


In [ ]:
with CATALOG_PATH.open('rb') as handle:
    bgs_matched = pickle.load(handle)

rm_tab = unique(bgs_matched, keys='ID')
print(f'Clusters: {len(rm_tab):,}')

cluster_ra = np.asarray(rm_tab['RA_x'], dtype=float)
cluster_dec = np.asarray(rm_tab['DEC_x'], dtype=float)
cluster_z = np.asarray(rm_tab['Z_SPEC_x'], dtype=float)


## General Plotting Utilities


In [ ]:
def describe(name, arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    q = np.percentile(arr, [5, 16, 50, 84, 95]) if len(arr) else np.full(5, np.nan)
    print(f'
{name}')
    print(f'  N      = {len(arr):,}')
    print(f'  mean   = {np.nanmean(arr):.4g}')
    print(f'  std    = {np.nanstd(arr):.4g}')
    print(f'  q05    = {q[0]:.4g}')
    print(f'  q16    = {q[1]:.4g}')
    print(f'  median = {q[2]:.4g}')
    print(f'  q84    = {q[3]:.4g}')
    print(f'  q95    = {q[4]:.4g}')


def print_correlation(label, x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if np.count_nonzero(mask) < 3:
        print(f'{label}: not enough finite points')
        return
    rp, pp = pearsonr(x[mask], y[mask])
    rs, ps = spearmanr(x[mask], y[mask])
    print(f'
{label}')
    print(f'  Pearson  r = {rp:.3f}, p = {pp:.3e}')
    print(f'  Spearman r = {rs:.3f}, p = {ps:.3e}')


def add_binned_mean(ax, x, y, nbins=8, label='binned mean'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0)
    x = x[mask]
    y = y[mask]
    if len(x) < nbins:
        return
    bins = np.logspace(np.log10(np.nanmin(x)), np.log10(np.nanmax(x)), nbins + 1)
    centers = np.sqrt(bins[:-1] * bins[1:])
    mean, _, _ = binned_statistic(x, y, statistic='mean', bins=bins)
    std, _, _ = binned_statistic(x, y, statistic='std', bins=bins)
    count, _, _ = binned_statistic(x, y, statistic='count', bins=bins)
    good = count > 3
    sem = std / np.sqrt(np.clip(count, 1, None))
    ax.errorbar(centers[good], mean[good], yerr=sem[good], fmt='o', color='crimson', capsize=3, label=label)


## Full sweep-file accumulation

This is the RAM-safe version of the one-file calculation. It reads one DR9 sweep file at a time, counts that file's contribution to the signal and background annuli, adds the counts to running totals, accumulates the sweep-file coverage area, then deletes the table before moving to the next file.

In [ ]:
def find_sweep_files(sweep_dirs=SWEEP_DIRS, pattern=SWEEP_PATTERN, max_files=MAX_SWEEP_FILES):
    files = []
    for sweep_dir in sweep_dirs:
        sweep_dir = Path(sweep_dir)
        files.extend(glob(str(sweep_dir / pattern)))
        files.extend(glob(str(sweep_dir / '*' / pattern)))
    files = sorted(files)
    if max_files is not None:
        files = files[:max_files]
    if len(files) == 0:
        raise FileNotFoundError(f'No sweep files matched {[str(Path(d) / pattern) for d in sweep_dirs]}')
    return [Path(f) for f in files]


def annulus_area_for_clusters(z_cl, Rmin_hMpc, Rmax_hMpc):
    z_cl = np.asarray(z_cl, dtype=float)
    area_sr = np.full(len(z_cl), np.nan, dtype=float)
    for i, z in enumerate(z_cl):
        if not np.isfinite(z) or z <= 0:
            continue
        th_min, th_max = annulus_theta_radians(z, Rmin_hMpc=Rmin_hMpc, Rmax_hMpc=Rmax_hMpc)
        area_sr[i] = spherical_annulus_area_sr(th_min, th_max)
    return area_sr, area_sr * (180.0 / np.pi) ** 2


def clusters_overlapping_sweep_box(ra_cl, dec_cl, z_cl, sweep_bounds, Rmax_hMpc):
    """Fast prefilter for clusters whose annulus could overlap the sweep box."""

    ra_cl = np.asarray(ra_cl, dtype=float)
    dec_cl = np.asarray(dec_cl, dtype=float)
    z_cl = np.asarray(z_cl, dtype=float)
    theta_max_deg = np.full(len(z_cl), np.nan, dtype=float)

    finite_z = np.isfinite(z_cl) & (z_cl > 0)
    theta_max_deg[finite_z] = [
        np.rad2deg(annulus_theta_radians(z, Rmax_hMpc=Rmax_hMpc)[1])
        for z in z_cl[finite_z]
    ]

    cos_dec = np.cos(np.deg2rad(dec_cl))
    ra_pad = theta_max_deg / np.clip(cos_dec, 0.2, None)
    dec_pad = theta_max_deg

    return (
        finite_z
        & (ra_cl >= sweep_bounds['ra_min'] - ra_pad)
        & (ra_cl <= sweep_bounds['ra_max'] + ra_pad)
        & (dec_cl >= sweep_bounds['dec_min'] - dec_pad)
        & (dec_cl <= sweep_bounds['dec_max'] + dec_pad)
    )


## All-Sweep Footprint And Density Diagnostics

This replaces the older one-file diagnostic. It streams every north+south DR9 sweep file one at a time, builds an all-sweep HEALPix galaxy surface-density map, overlays the cluster positions, and saves per-file density statistics. No full DR9 galaxy table is held in memory.


In [ ]:
sweep_files = find_sweep_files()
print(f'Found {len(sweep_files):,} sweep files')
print('First file:', sweep_files[0])
print('Last file:', sweep_files[-1])

NSIDE_SWEEP_DENSITY = 256
npix_density = hp.nside2npix(NSIDE_SWEEP_DENSITY)
pix_area_deg2 = hp.nside2pixarea(NSIDE_SWEEP_DENSITY, degrees=True)
galaxy_counts_map = np.zeros(npix_density, dtype=np.int64)

file_rows = []
for i_file, sweep_file in enumerate(sweep_files, start=1):
    print(f'[{i_file}/{len(sweep_files)}] density diagnostic: {sweep_file.name}')
    dr9_i = read_one_dr9_sweep(sweep_file, r_mag_limit=R_MAG_LIMIT)
    ra_i = np.asarray(dr9_i['RA'], dtype=float)
    dec_i = np.asarray(dr9_i['DEC'], dtype=float)
    finite = np.isfinite(ra_i) & np.isfinite(dec_i)

    if np.any(finite):
        pix_i = hp.ang2pix(
            NSIDE_SWEEP_DENSITY,
            np.deg2rad(90.0 - dec_i[finite]),
            np.deg2rad(ra_i[finite]),
        )
        np.add.at(galaxy_counts_map, pix_i, 1)
        occupied_pix = np.unique(pix_i)
        occupied_area_deg2 = len(occupied_pix) * pix_area_deg2
        selected_density = np.count_nonzero(finite) / occupied_area_deg2 if occupied_area_deg2 > 0 else np.nan
        file_rows.append({
            'filename': str(sweep_file),
            'n_selected': int(np.count_nonzero(finite)),
            'occupied_area_deg2': float(occupied_area_deg2),
            'selected_density_deg2': float(selected_density),
            'ra_min': float(np.nanmin(ra_i[finite])),
            'ra_max': float(np.nanmax(ra_i[finite])),
            'dec_min': float(np.nanmin(dec_i[finite])),
            'dec_max': float(np.nanmax(dec_i[finite])),
        })
    else:
        file_rows.append({
            'filename': str(sweep_file),
            'n_selected': 0,
            'occupied_area_deg2': 0.0,
            'selected_density_deg2': np.nan,
            'ra_min': np.nan,
            'ra_max': np.nan,
            'dec_min': np.nan,
            'dec_max': np.nan,
        })

    del dr9_i
    gc.collect()

file_density_table = Table(rows=file_rows)
file_density_table.write(OUTPUT_DIR / 'dr9_all_sweeps_file_density_diagnostic.ecsv', format='ascii.ecsv', overwrite=True)
print('Saved per-file density diagnostic:', OUTPUT_DIR / 'dr9_all_sweeps_file_density_diagnostic.ecsv')


In [ ]:
density_map_deg2 = galaxy_counts_map.astype(float) / pix_area_deg2
density_plot = density_map_deg2.copy()
density_plot[galaxy_counts_map == 0] = hp.UNSEEN

finite_density = density_map_deg2[galaxy_counts_map > 0]
if len(finite_density) > 0:
    vmin, vmax = np.nanpercentile(finite_density, [2, 98])
else:
    vmin, vmax = None, None

fig = plt.figure(figsize=(11, 6.5))
hp.mollview(
    density_plot,
    fig=fig.number,
    title='DR9 selected galaxy surface density, all north+south sweeps',
    unit=r'galaxies deg$^{-2}$',
    min=vmin,
    max=vmax,
    cmap='viridis',
)
hp.projscatter(
    cluster_ra,
    cluster_dec,
    lonlat=True,
    s=2,
    alpha=0.35,
    color='crimson',
    label='redMaPPer/BGS clusters',
)
hp.graticule(color='white', alpha=0.25)
fig.savefig(OUTPUT_DIR / 'dr9_all_sweeps_mollview_density_clusters.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
good_density = np.isfinite(file_density_table['selected_density_deg2']) & (file_density_table['selected_density_deg2'] > 0)
ax.hist(np.asarray(file_density_table['selected_density_deg2'][good_density], dtype=float), bins=50, histtype='stepfilled', alpha=0.75)
ax.set_xlabel(r'per-file selected galaxy density [deg$^{-2}$]')
ax.set_ylabel('number of sweep files')
ax.set_title('DR9 sweep-to-sweep density variation')
fig.savefig(OUTPUT_DIR / 'dr9_all_sweeps_file_density_histogram.png', dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.scatter(
    file_density_table['occupied_area_deg2'],
    file_density_table['n_selected'],
    s=12,
    alpha=0.6,
)
ax.set_xlabel(r'occupied HEALPix area [deg$^2$]')
ax.set_ylabel('selected DR9 galaxies')
ax.set_title('Selected galaxies per sweep file')
fig.savefig(OUTPUT_DIR / 'dr9_all_sweeps_file_counts_vs_area.png', dpi=180)
plt.show()


In [ ]:
sweep_files = find_sweep_files()
print(f'Found {len(sweep_files):,} sweep files')
print('First file:', sweep_files[0])
print('Last file:', sweep_files[-1])

n_cl = len(rm_tab)
ra_cl = np.asarray(rm_tab['RA_x'], dtype=float)
dec_cl = np.asarray(rm_tab['DEC_x'], dtype=float)
z_cl = np.asarray(rm_tab['Z_SPEC_x'], dtype=float)

area_signal_sr_all, area_signal_deg2_all = annulus_area_for_clusters(z_cl, RMIN_HMPC, RMAX_HMPC)
area_bg_sr_all, area_bg_deg2_all = annulus_area_for_clusters(z_cl, BG_RMIN_HMPC, BG_RMAX_HMPC)

Ngal_signal_all = np.zeros(n_cl, dtype=np.int64)
Ngal_bg_all = np.zeros(n_cl, dtype=np.int64)
covered_area_signal_deg2 = np.zeros(n_cl, dtype=float)
covered_area_bg_deg2 = np.zeros(n_cl, dtype=float)
n_files_touching_cluster = np.zeros(n_cl, dtype=np.int16)

for i_file, sweep_file in enumerate(sweep_files, start=1):
    print(f'\n[{i_file}/{len(sweep_files)}] {sweep_file.name}')

    sweep_bounds_i = read_sweep_bounds(sweep_file)
    overlap = clusters_overlapping_sweep_box(
        ra_cl,
        dec_cl,
        z_cl,
        sweep_bounds_i,
        Rmax_hMpc=BG_RMAX_HMPC,
    )
    idx = np.where(overlap)[0]
    print(f'  clusters with possible overlap: {len(idx):,}')

    if len(idx) == 0:
        continue

    dr9_i = read_one_dr9_sweep(sweep_file, r_mag_limit=R_MAG_LIMIT)
    print(f'  selected galaxies: {len(dr9_i):,}')

    Nsig_i, _ = count_galaxies_in_annuli(
        ra_cl[idx],
        dec_cl[idx],
        z_cl[idx],
        dr9_i['RA'],
        dr9_i['DEC'],
        Rmin_hMpc=RMIN_HMPC,
        Rmax_hMpc=RMAX_HMPC,
        nside=NSIDE_GAL,
        verbose_every=0,
    )
    Nbg_i, _ = count_galaxies_in_annuli(
        ra_cl[idx],
        dec_cl[idx],
        z_cl[idx],
        dr9_i['RA'],
        dr9_i['DEC'],
        Rmin_hMpc=BG_RMIN_HMPC,
        Rmax_hMpc=BG_RMAX_HMPC,
        nside=NSIDE_GAL,
        verbose_every=0,
    )

    cov_sig_i, _ = sweep_annulus_coverage_for_clusters(
        ra_cl[idx],
        dec_cl[idx],
        z_cl[idx],
        sweep_bounds_i,
        Rmin_hMpc=RMIN_HMPC,
        Rmax_hMpc=RMAX_HMPC,
        nside=NSIDE_SWEEP_COVERAGE,
        verbose_every=0,
    )
    cov_bg_i, _ = sweep_annulus_coverage_for_clusters(
        ra_cl[idx],
        dec_cl[idx],
        z_cl[idx],
        sweep_bounds_i,
        Rmin_hMpc=BG_RMIN_HMPC,
        Rmax_hMpc=BG_RMAX_HMPC,
        nside=NSIDE_SWEEP_COVERAGE,
        verbose_every=0,
    )

    Ngal_signal_all[idx] += Nsig_i
    Ngal_bg_all[idx] += Nbg_i
    covered_area_signal_deg2[idx] += np.nan_to_num(cov_sig_i, nan=0.0) * area_signal_deg2_all[idx]
    covered_area_bg_deg2[idx] += np.nan_to_num(cov_bg_i, nan=0.0) * area_bg_deg2_all[idx]
    n_files_touching_cluster[idx] += 1

    del dr9_i, Nsig_i, Nbg_i, cov_sig_i, cov_bg_i
    gc.collect()

    if SAVE_EVERY_N_FILES and (i_file % SAVE_EVERY_N_FILES == 0):
        print('  saving checkpoint')
        checkpoint = Table()
        checkpoint['ID'] = rm_tab['ID']
        checkpoint['Ngal_signal_DR9_annulus'] = Ngal_signal_all
        checkpoint['Ngal_bg_DR9_annulus'] = Ngal_bg_all
        checkpoint['covered_area_signal_deg2'] = covered_area_signal_deg2
        checkpoint['covered_area_bg_deg2'] = covered_area_bg_deg2
        checkpoint['n_files_touching_cluster'] = n_files_touching_cluster
        checkpoint.write(OUTPUT_DIR / 'dr9_all_sweeps_running_checkpoint.ecsv', format='ascii.ecsv', overwrite=True)

print('\nFinished all sweep files')


In [ ]:
coverage_signal_all = covered_area_signal_deg2 / area_signal_deg2_all
coverage_bg_all = covered_area_bg_deg2 / area_bg_deg2_all

# If neighboring sweep boxes overlap, coverage can slightly exceed one. Keep the
# raw accumulated value for debugging, but use a clipped value for cuts/plots.
coverage_signal_clipped = np.clip(coverage_signal_all, 0.0, 1.0)
coverage_bg_clipped = np.clip(coverage_bg_all, 0.0, 1.0)

Sigma_signal_all = Ngal_signal_all / covered_area_signal_deg2
Sigma_bg_all = Ngal_bg_all / covered_area_bg_deg2
Sigma_excess_local_all = Sigma_signal_all - Sigma_bg_all
Nexcess_local_all = Sigma_excess_local_all * area_signal_deg2_all

# Attach the new DR9 local-overdensity columns directly to a copy of the
# redMaPPer cluster table.  This is the final cluster table to use downstream.
rm_tab_dr9 = rm_tab.copy()
rm_tab_dr9['Ngal_signal_DR9_annulus'] = Ngal_signal_all
rm_tab_dr9['Ngal_bg_DR9_annulus'] = Ngal_bg_all
rm_tab_dr9['area_signal_deg2'] = area_signal_deg2_all
rm_tab_dr9['area_bg_deg2'] = area_bg_deg2_all
rm_tab_dr9['covered_area_signal_deg2'] = covered_area_signal_deg2
rm_tab_dr9['covered_area_bg_deg2'] = covered_area_bg_deg2
rm_tab_dr9['coverage_signal_sweep'] = coverage_signal_clipped
rm_tab_dr9['coverage_bg_sweep'] = coverage_bg_clipped
rm_tab_dr9['coverage_signal_sweep_raw'] = coverage_signal_all
rm_tab_dr9['coverage_bg_sweep_raw'] = coverage_bg_all
rm_tab_dr9['n_files_touching_cluster'] = n_files_touching_cluster
rm_tab_dr9['Sigma_signal_covcorr'] = Sigma_signal_all
rm_tab_dr9['Sigma_bg_covcorr'] = Sigma_bg_all
rm_tab_dr9['Sigma_excess_local_DR9_annulus'] = Sigma_excess_local_all
rm_tab_dr9['Nexcess_local_DR9_annulus'] = Nexcess_local_all

# Keep the older variable name as an alias so downstream cells still work.
rm_all_sweeps = rm_tab_dr9

full_mask = (
    (np.asarray(rm_all_sweeps['coverage_signal_sweep'], dtype=float) > COVERAGE_MIN)
    & (np.asarray(rm_all_sweeps['coverage_bg_sweep'], dtype=float) > COVERAGE_MIN)
    & np.isfinite(rm_all_sweeps['Nexcess_local_DR9_annulus'])
    & np.isfinite(rm_all_sweeps['Sigma_excess_local_DR9_annulus'])
)
rm_all_sweeps_filt = rm_all_sweeps[full_mask]

print(f'All-sweep coverage cut: signal and background sweep coverage > {COVERAGE_MIN}')
print(f'Kept {len(rm_all_sweeps_filt):,}/{len(rm_all_sweeps):,} clusters')
print(f"Clusters with Ngal_signal > 0 before coverage cut: {np.count_nonzero(np.asarray(rm_all_sweeps['Ngal_signal_DR9_annulus']) > 0):,}")
print(f"Clusters with Ngal_signal > 0 after coverage cut: {np.count_nonzero(np.asarray(rm_all_sweeps_filt['Ngal_signal_DR9_annulus']) > 0):,}")

if np.nanmax(coverage_signal_all) > 1.05 or np.nanmax(coverage_bg_all) > 1.05:
    print('Warning: some accumulated sweep coverages exceed 1.05; check for overlapping sweep files or duplicated regions.')

# Save the redMaPPer cluster table with the new DR9 local-overdensity columns.
rm_tab_dr9.write(OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep.ecsv', format='ascii.ecsv', overwrite=True)
rm_tab_dr9.write(OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep.fits', overwrite=True)
rm_all_sweeps_filt.write(OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep_coverage_cut.ecsv', format='ascii.ecsv', overwrite=True)
rm_all_sweeps_filt.write(OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep_coverage_cut.fits', overwrite=True)

# Backward-compatible names from the first draft of this notebook.
rm_tab_dr9.write(OUTPUT_DIR / 'dr9_all_sweeps_cluster_counts.ecsv', format='ascii.ecsv', overwrite=True)
rm_all_sweeps_filt.write(OUTPUT_DIR / 'dr9_all_sweeps_cluster_counts_coverage_cut.ecsv', format='ascii.ecsv', overwrite=True)

print('Saved redMaPPer table with DR9 local-overdensity columns to:')
print(' ', OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep.ecsv')
print(' ', OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep.fits')
print(' ', OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep_coverage_cut.ecsv')
print(' ', OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep_coverage_cut.fits')


In [ ]:
x = np.asarray(rm_all_sweeps_filt['LAMBDA'], dtype=float)
y = np.asarray(rm_all_sweeps_filt['Nexcess_local_DR9_annulus'], dtype=float)
coverage_plot = np.asarray(rm_all_sweeps_filt['coverage_signal_sweep'], dtype=float)

fig, ax = plt.subplots(figsize=(6.8, 5.2))
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(coverage_plot)
sc = ax.scatter(x[mask], y[mask], c=coverage_plot[mask], s=18, alpha=0.65)
add_binned_mean(ax, x[mask], y[mask], nbins=8, label='binned mean')
ax.set_xscale('log')
ax.set_xlabel(r'$\lambda_{\rm RM}$')
ax.set_ylabel(r'$N_{\rm excess,local}$')
ax.set_title(fr'All DR9 sweeps: signal ${RMIN_HMPC}<R<{RMAX_HMPC}$, background ${BG_RMIN_HMPC}<R<{BG_RMAX_HMPC}\ h^{{-1}}\mathrm{{Mpc}}$')
ax.legend(frameon=False)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('signal sweep coverage')
fig.savefig(OUTPUT_DIR / 'dr9_all_sweeps_local_background_richness_relation.png', dpi=180)
plt.show()

print_correlation('All sweeps: local-background DR9 excess count vs richness', x, y)
print_correlation('All sweeps: local-background DR9 excess surface density vs richness', x, rm_all_sweeps_filt['Sigma_excess_local_DR9_annulus'])


## Compare local overdensity with spectroscopic richness

This section uses the redMaPPer/local-overdensity table after `lambda_spec` has been joined from `Scaling_Relations_Fit.ipynb`. It applies the same sweep-coverage cut and plots/correlates local-overdensity statistics against spectroscopic richness.

In [ ]:
from astropy.table import Table

lambda_spec_overdensity_path = OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep_with_lambda_spec.fits'
if not lambda_spec_overdensity_path.exists():
    raise FileNotFoundError(
        f'Missing {lambda_spec_overdensity_path}. Run the final lambda_spec join cell in '
        'Scaling_Relations_Fit.ipynb first.'
    )

rm_overdensity_spec = Table.read(lambda_spec_overdensity_path)
print(f'Loaded {lambda_spec_overdensity_path}')
print(f'Rows: {len(rm_overdensity_spec):,}')


def col_float(table, col):
    """Return a column as a float ndarray, filling masked values with NaN."""

    arr = np.ma.asarray(table[col], dtype=float)
    return np.ma.filled(arr, np.nan)


lambda_spec = col_float(rm_overdensity_spec, 'lambda_spec')
coverage_signal = col_float(rm_overdensity_spec, 'coverage_signal_sweep')
coverage_bg = col_float(rm_overdensity_spec, 'coverage_bg_sweep')
z_spec = col_float(rm_overdensity_spec, 'Z_SPEC_x')

base_mask = (
    np.isfinite(lambda_spec)
    & (lambda_spec > 0)
    & np.isfinite(coverage_signal)
    & np.isfinite(coverage_bg)
    & (coverage_signal > COVERAGE_MIN)
    & (coverage_bg > COVERAGE_MIN)
)

print(f'Clusters with finite lambda_spec: {np.count_nonzero(np.isfinite(lambda_spec) & (lambda_spec > 0)):,}')
print(f'Clusters after lambda_spec + coverage cut: {np.count_nonzero(base_mask):,}')


In [ ]:
overdensity_columns = [
    ('Ngal_signal_DR9_annulus', r'$N_{\rm signal}$'),
    ('Ngal_bg_DR9_annulus', r'$N_{\rm bg}$'),
    ('Sigma_signal_covcorr', r'$\Sigma_{\rm signal}$'),
    ('Sigma_bg_covcorr', r'$\Sigma_{\rm bg}$'),
    ('Sigma_excess_local_DR9_annulus', r'$\Sigma_{\rm excess,local}$'),
    ('Nexcess_local_DR9_annulus', r'$N_{\rm excess,local}$'),
]

correlation_rows = []

for col, label in overdensity_columns:
    y = col_float(rm_overdensity_spec, col)
    mask = base_mask & np.isfinite(y)

    if np.count_nonzero(mask) < 3:
        correlation_rows.append((col, np.count_nonzero(mask), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan))
        continue

    pearson_r, pearson_p = pearsonr(lambda_spec[mask], y[mask])
    spearman_r, spearman_p = spearmanr(lambda_spec[mask], y[mask])
    pearson_logx_r, pearson_logx_p = pearsonr(np.log10(lambda_spec[mask]), y[mask])

    correlation_rows.append((
        col,
        np.count_nonzero(mask),
        pearson_r,
        pearson_p,
        pearson_logx_r,
        pearson_logx_p,
        spearman_r,
        spearman_p,
    ))

correlation_table = Table(
    rows=correlation_rows,
    names=[
        'quantity',
        'N',
        'pearson_r_lambda_spec',
        'pearson_p_lambda_spec',
        'pearson_r_log_lambda_spec',
        'pearson_p_log_lambda_spec',
        'spearman_r_lambda_spec',
        'spearman_p_lambda_spec',
    ],
)

correlation_table.write(
    OUTPUT_DIR / 'lambda_spec_local_overdensity_correlations.ecsv',
    format='ascii.ecsv',
    overwrite=True,
)

correlation_table


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.ravel()

for ax, (col, ylabel) in zip(axes, overdensity_columns):
    y = col_float(rm_overdensity_spec, col)
    mask = base_mask & np.isfinite(y) & np.isfinite(z_spec)

    sc = ax.scatter(
        lambda_spec[mask],
        y[mask],
        c=z_spec[mask],
        s=14,
        alpha=0.55,
        cmap='viridis',
        edgecolor='none',
    )

    add_binned_mean(ax, lambda_spec[mask], y[mask], nbins=8, label='binned mean')

    if 'excess' in col.lower():
        ax.axhline(0.0, color='black', lw=1.0, alpha=0.55)

    ax.set_xscale('log')
    ax.set_xlabel(r'$\lambda_{\rm spec}$')
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle(
    fr'Local overdensity versus spectroscopic richness; coverage $>{COVERAGE_MIN}$',
    fontsize=15,
)
cbar = fig.colorbar(sc, ax=axes.tolist(), pad=0.01)
cbar.set_label(r'$z_{\rm spec,BCG}$')
fig.savefig(OUTPUT_DIR / 'lambda_spec_local_overdensity_scatter_grid.png', dpi=180)
plt.show()


In [ ]:
# Focused publication-style view of the preferred estimator.
preferred_col = 'Nexcess_local_DR9_annulus'
y = col_float(rm_overdensity_spec, preferred_col)
mask = base_mask & np.isfinite(y) & np.isfinite(z_spec)

fig, ax = plt.subplots(figsize=(7.2, 5.4))
sc = ax.scatter(
    lambda_spec[mask],
    y[mask],
    c=z_spec[mask],
    s=18,
    alpha=0.6,
    cmap='viridis',
    edgecolor='none',
)
add_binned_mean(ax, lambda_spec[mask], y[mask], nbins=8, label='binned mean')
ax.axhline(0.0, color='black', lw=1.0, alpha=0.6)
ax.set_xscale('log')
ax.set_xlabel(r'$\lambda_{\rm spec}$')
ax.set_ylabel(r'$N_{\rm excess,local}$')

rs, ps = spearmanr(lambda_spec[mask], y[mask])
rp, pp = pearsonr(np.log10(lambda_spec[mask]), y[mask])
ax.text(
    0.04,
    0.96,
    rf'Spearman $r_s={rs:.2f}$, $p={ps:.1e}$' + '\n' + rf'Pearson $(\log\lambda_{{\rm spec}}, y)$ $r={rp:.2f}$, $p={pp:.1e}$',
    transform=ax.transAxes,
    ha='left',
    va='top',
    fontsize=11,
    bbox=dict(facecolor='white', edgecolor='none', alpha=0.8),
)
ax.legend(frameon=False)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'$z_{\rm spec,BCG}$')
fig.savefig(OUTPUT_DIR / 'lambda_spec_vs_Nexcess_local_DR9.png', dpi=180)
plt.show()

print_correlation('Preferred estimator: N_excess_local vs lambda_spec', lambda_spec[mask], y[mask])
